<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/02-rdd-fundamentals/05-broadcast-accumulators.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 2.5 — Broadcast variables and accumulators

Ends with the double-counting demo — the module's concepts (laziness,
re-execution, caching) colliding in one observable bug.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("2.5-broadcast-accumulators")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

lines = sc.textFile(f"{DATA_DIR}/sample_logs.txt")

## Broadcast: ship a lookup table once per executor

In [ ]:
team_lookup = {"auth": "Identity", "api": "Gateway", "payments": "Billing"}
bteams = sc.broadcast(team_lookup)

per_team = (
    lines.map(lambda l: (bteams.value.get(l.split()[3], "Unknown"), 1))
         .reduceByKey(lambda a, b: a + b)
)
sorted(per_team.collect())

In [ ]:
# Read-only in tasks: this mutation happens on a task-local copy and is lost
def try_mutate(l):
    bteams.value["auth"] = "HACKED"      # no error, no effect beyond this task
    return l

lines.map(try_mutate).count()
print("driver copy still:", bteams.value["auth"])

bteams.unpersist()   # free executor copies when done

## Accumulators: count bad records during the pass you were already making

In [ ]:
bad_lines = sc.accumulator(0)

def parse_latency(l):
    global bad_lines
    parts = dict(kv.split("=") for kv in l.split()[4:] if "=" in kv)
    if "latency_ms" not in parts:
        bad_lines += 1                  # tasks may only ADD
        return None
    return int(parts["latency_ms"])

latencies = lines.map(parse_latency).filter(lambda x: x is not None)

print("before any action:", bad_lines.value)   # 0 — laziness (lesson 2.1)
print("max latency:", latencies.max())
print("after the action: ", bad_lines.value)   # lines without latency_ms

## The double-count bug, live

Nothing is cached, so the second action re-runs the `map` — and the
accumulator keeps adding.

In [ ]:
print("after 1st action:", bad_lines.value)
latencies.count()                                # lineage re-runs!
print("after 2nd action:", bad_lines.value, "<- double-counted")
latencies.take(3)
print("after 3rd action:", bad_lines.value, "<- and again")

In [ ]:
# Fix: cache, so later actions start from materialized data instead of re-running the map
bad_lines2 = sc.accumulator(0)

def parse2(l):
    global bad_lines2
    parts = dict(kv.split("=") for kv in l.split()[4:] if "=" in kv)
    if "latency_ms" not in parts:
        bad_lines2 += 1
        return None
    return int(parts["latency_ms"])

lat2 = lines.map(parse2).filter(lambda x: x is not None).cache()
lat2.count()
print("after 1st action:", bad_lines2.value)
lat2.count(); lat2.take(3)
print("after more actions:", bad_lines2.value, "<- stable now")

## Exercises

1. Build a broadcast set of "VIP users" `{'u101','u105','u110'}` and count log events from VIPs vs non-VIPs in one `reduceByKey` pass.
2. Add a second accumulator counting WARN lines inside the same `parse` function — verify both totals after one action, using the cached version.
3. Predict, then verify: with the CACHED `lat2`, does `lat2.filter(lambda x: x > 100).count()` change `bad_lines2`? Why or why not?

In [ ]:
# your exercise code here
